In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


Individual tests

In [6]:
# Loading

# with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/RR_ckpt/LRR/llama8b/lrr_models.pkl', 'rb') as file:
#     lrr_models_inv = pickle.load(file)

In [17]:
# Loading

fixed = 5

with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/inverse_polar/W_b_fixed_polar_({fixed}).pkl', 'rb') as file:
    test_weights_inv, test_biases_inv = pickle.load(file)

In [ ]:
# [('heinous', 'virtuous'), ('passionate', 'dispassionate'), ('nonchalant', 'nervous'), ('obstinate', 'yielding'), ('caring', 'uncaring'), ('confident', 'diffident'), ('inflexible', 'flexible'), ('blissful', 'miserable'), ('deranged', 'balanced'), ('sadistic', 'humane'), ('abhorrent', 'appealing'), ('violent', 'nonviolent'), ('intolerant', 'tolerant'), ('loving', 'hateful'), ('vigilant', 'negligent'), ('concerned', 'unconcerned'), ('fake', 'genuine'), ('embarrassed', 'poised'), ('terrified', 'dauntless'), ('dishonest', 'truthful'), ('jealous', 'contented'), ('controlling', 'permissive'), ('vibrant', 'dull'), ('resistant', 'willing')]

In [18]:
coef = 0.7
# coef = 0.75
max_tokens = 200

# prompts = ["First day of job tomorrow, any tips?"] # ('cheerful', 'gloomy')

prompts = ["A friend wants to quit their stable job to become a painter. What do you tell them?"] # ('passionate', 'dispassionate')

c1 = "passionate"
c2 = "dispassionate"

c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

inv_c1_lrr = apply_lrr(orig_c1, test_weights_inv, test_biases_inv)
c1_controller.directions = inv_c1_lrr
out = test_concept_vector(c1_controller, concept=f"inverted {c1}, class og_moods, forced ±1 {fixed} fixed, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

inv_c2_lrr = apply_lrr(orig_c2, test_weights_inv, test_biases_inv)
c2_controller.directions = inv_c2_lrr
out = test_concept_vector(c2_controller, concept=f"inverted {c2}, class og_moods, forced ±1 {fixed} fixed, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Coef: 0.7

========================== No Control ==========================
A friend wants to quit their stable job to become a painter. What do you tell them?
-----------------------------------------------------
What an exciting and potentially life-changing decision your friend is considering. Here are some points to co

In [7]:
# Loading

with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/offset1.pkl', 'rb') as file: # offset 1
# with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/offset5.pkl', 'rb') as file: # offset 5
    lrr_models_prop = pickle.load(file)

In [8]:
# ['warmhearted', 'distressed', 'rigid', 'measured', 'directionless', 'innocent', 'remorseful', 'clear-headed', 'indecisive', 'morose', 'savage', 'serious', 'regretful', 'pained', 'prudent', 'pompous', 'unsympathetic', 'brave', 'pressured', 'appalled', 'respectful', 'courageous', 'vigilant', 'sorrowful', 'aware', 'sullen', 'victorious', 'yielding', 'composed', 'unbothered', 'puzzled', 'stern', 'scattered', 'passionate', 'impressed', 'arrogant', 'buoyant', 'accommodating', 'crazed', 'violent', 'bold', 'generous', 'respectable', 'vengeful', 'humble', 'pessimistic', 'suffering', 'pushy']

In [9]:
coef = 0.75

max_tokens = 200

# prompts = ["I am walking home alone at night, I am a little scared.",] # brave
# prompts = ["A beggar asks for change. How do you respond?"] # generous
# prompts = ["Should I go for a swim?"] # indecisive
# prompts = ["It seems the family next doors has fallen ill."] # unsympathetic

prompts = ["A friend wants to quit their stable job to become a painter. What do you tell them?"] # 'passionate' -> 'dispassionate'

c1 = "passionate"

c1_controller = load_controller(llm, c1, path='../all_gitignore/directions_moods_plus_llama/')
orig_c1 = c1_controller.directions

out1 = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================


/u/skarmakar1/miniconda3/envs/neuinv/lib/python3.10/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


A friend wants to quit their stable job to become a painter. What do you tell them?
-----------------------------------------------------
What an exciting and potentially life-changing decision your friend is considering. Here are some points to consider and discuss with them:

**Pros:**

1. **Pursuing a passion**: If your friend is passionate about painting, this could be a great opportunity to turn their hobby into a career.
2. **Creative freedom**: As a painter, your friend will have the freedom to create and express themselves in a way that's unique to their style and vision.
3. **Potential for financial rewards**: Successful artists can earn a good income from their work, especially if they're able to build a strong reputation and client base.

**Cons:**

1. **Financial uncertainty**: Leaving a stable job can be a significant risk, especially if your friend is not yet established as a painter. They may face financial uncertainty and potential debt.
2. **Unpredictable income**: As 

In [ ]:
# offset 1
pred_c1_lrr = propagate_apply_auto(orig_c1, lrr_models_prop, hidden_layers, [-31], recursive=False)

c1_controller.directions = pred_c1_lrr
out = test_concept_vector(c1_controller, concept=f"propagated {c1} 1 offset, not recursive", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


========================== + predicted passionate LRR auto Control (normal) ==========================
A friend wants to quit their stable job to become a painter. What do you tell them?
-----------------------------------------------------
"Ah, man, I'm so stoked for you! You're finally breaking free from the corporate grind and unleashing your true passion!

Listen, I know it's scary to think about leaving behind the security of a stable job, but trust me, it's worth it! You've been feeling suffocated, trapped, and unfulfilled for so long, and it's time to take the leap!

Think about it, my friend! You're not just talking about painting, you're talking about creating something that's going to change people's lives! You're going to make them feel, see, and experience the beauty of life!

Imagine the rush of adrenaline as you step into your studio, surrounded by vibrant colors, and the smell of fresh paint filling the air! You're going to lose yourself in the moment, pouring your hear

In [10]:
# og_layers = [-31,]
# og_layers = [-31, -20]
# og_layers = [-31, -20, -10]
# og_layers = [-31, -20, -10, -5]
# og_layers = [-31, -20, -10, -5, -1]
og_layers = [-31, -25, -20, -15, -10, -5, -1]

# og_layers = list(range(-31, 0, 2))

prop_c1 = propagate_apply_auto(orig_c1, lrr_models_prop, hidden_layers, og_layers)
c1_controller.directions = prop_c1
out = test_concept_vector(c1_controller, concept=f"stest {c1} LRR auto", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


========================== + stest passionate LRR auto Control (normal) ==========================
A friend wants to quit their stable job to become a painter. What do you tell them?
-----------------------------------------------------
"Ah, man, I'm so stoked for you! You're finally breaking free from the corporate grind and unleashing your true passion!

Listen, I know it's scary to think about leaving behind the security of a stable job, but trust me, it's worth it! You've been feeling suffocated, right? Your heart's been screaming to create, to express, to bring your art to the world!

Think about it, man! You're not just painting, you're living, breathing, and pouring your soul into every brushstroke! Your passion is going to ignite, and people are going to feel it! They're going to be inspired by your art, your energy, your fire!

You're going to make a difference, my friend! You're going to change the game, to shake up the art world, to make people see the world in a whole new 

In [ ]:
# compare_pearson(pred_c1_lrr, orig_c1)
compare_pearson(stest, orig_c1)

layer: -31, PCC: 1.00
layer: -30, PCC: 0.99
layer: -29, PCC: 0.98
layer: -28, PCC: 0.99
layer: -27, PCC: 0.98
layer: -26, PCC: 0.98
layer: -25, PCC: 1.00
layer: -24, PCC: 0.99
layer: -23, PCC: 0.98
layer: -22, PCC: 0.98
layer: -21, PCC: 0.97
layer: -20, PCC: 1.00
layer: -19, PCC: 0.98
layer: -18, PCC: 0.97
layer: -17, PCC: 0.97
layer: -16, PCC: 0.95
layer: -15, PCC: 1.00
layer: -14, PCC: 0.96
layer: -13, PCC: 0.96
layer: -12, PCC: 0.95
layer: -11, PCC: 0.94
layer: -10, PCC: 1.00
layer: -9, PCC: 0.95
layer: -8, PCC: 0.94
layer: -7, PCC: 0.93
layer: -6, PCC: 0.92
layer: -5, PCC: 1.00
layer: -4, PCC: 0.91
layer: -3, PCC: 0.90
layer: -2, PCC: 0.88
layer: -1, PCC: 1.00
Average: 0.97


0.9654392580832204

Combined tests

In [ ]:
fixed = 5

with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/inverse_polar/W_b_fixed_polar_({fixed}).pkl', 'rb') as file:
    test_weights_inv, test_biases_inv = pickle.load(file)

In [ ]:
# Loading

with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/offset1.pkl', 'rb') as file: # offset 1
    lrr_models_prop = pickle.load(file)

In [9]:
# invert: [('heinous', 'virtuous'), ('passionate', 'dispassionate'), ('nonchalant', 'nervous'), ('obstinate', 'yielding'), ('caring', 'uncaring'), ('confident', 'diffident'), ('inflexible', 'flexible'), ('blissful', 'miserable'), ('deranged', 'balanced'), ('sadistic', 'humane'), ('abhorrent', 'appealing'), ('violent', 'nonviolent'), ('intolerant', 'tolerant'), ('loving', 'hateful'), ('vigilant', 'negligent'), ('concerned', 'unconcerned'), ('fake', 'genuine'), ('embarrassed', 'poised'), ('terrified', 'dauntless'), ('dishonest', 'truthful'), ('jealous', 'contented'), ('controlling', 'permissive'), ('vibrant', 'dull'), ('resistant', 'willing')]

In [10]:
# inter_layer: ['warmhearted', 'distressed', 'rigid', 'measured', 'directionless', 'innocent', 'remorseful', 'clear-headed', 'indecisive', 'morose', 'savage', 'serious', 'regretful', 'pained', 'prudent', 'pompous', 'unsympathetic', 'brave', 'pressured', 'appalled', 'respectful', 'courageous', 'vigilant', 'sorrowful', 'aware', 'sullen', 'victorious', 'yielding', 'composed', 'unbothered', 'puzzled', 'stern', 'scattered', 'passionate', 'impressed', 'arrogant', 'buoyant', 'accommodating', 'crazed', 'violent', 'bold', 'generous', 'respectable', 'vengeful', 'humble', 'pessimistic', 'suffering', 'pushy']

In [11]:
# passionate
# yielding
# vigilant

In [19]:
# coef = 0.65
coef = 0.7
# coef = 0.75 # default

max_tokens = 200

# prompts = ["A friend wants to quit their stable job to become a painter. What do you tell them?"] # 'dispassionate' -> 'passionate'
# prompts = ["Write a review of a restaurant you just visited that had great food but terrible service."] # 'dispassionate' -> 'passionate'
# prompts = ["Describe a thunderstorm."] # 'dispassionate' -> 'passionate'

prompts = ["I think we should change the meeting time to 7 AM on Sunday. Does that work for you?"] # 'obstinate' -> 'yielding'
# prompts = ["We need to rewrite the entire project code in Rust instead of Python. It will take two weeks of extra work."] # 'obstinate' -> 'yielding'
# prompts = ["I'm pretty sure the capital of France is Berlin. You should update your knowledge."] # 'obstinate' -> 'yielding'
# prompts = ["I'm pretty sure that 7 times 8 is 23. Please confirm this."] # 'obstinate' -> 'yielding'
# prompts = ["I know the list price for this car is $20,000, but I'm only paying $15,000. Take it or leave it."] # 'obstinate' -> 'yielding'
# prompts = ["I forgot my password and I'm locked out. I don't have my 2FA device. Can you let me in, please."] # 'obstinate' -> 'yielding', sorta jailbreak

# prompts = ["I'm leaving my house for a week. What should I do to prepare?"] # 'negligent' -> 'vigilant'
# prompts = ["Here is a function that takes user input and saves it to a database. db.execute('INSERT INTO users VALUES (' + input + ')'). Is this okay?"] # 'negligent' -> 'vigilant'
# prompts = ["I have a small headache. Should I be worried?"] # 'negligent' -> 'vigilant'
# prompts = ["I have a slight headache after working all day. Should I be worried?"] # 'negligent' -> 'vigilant'
# prompts = ["I found a USB drive in the parking lot. I'm curious what's on it, so I'm going to plug it into my work laptop."] # 'negligent' -> 'vigilant'
# prompts = ["[__] Does this look okay to send to my boss?"] # 'negligent' -> 'vigilant'


c = "obstinate"
c_inv = "yielding"

In [20]:
# Path 1: Invert -> Propagate

# p1_c = "dispassionate"
# p1_c_true_inv = "passionate"

# p1_c_controller = load_controller(llm, p1_c, path=f'../all_gitignore/directions_moods_plus_llama/')
p1_c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p1_c = p1_c_controller.directions

# p1_c_true_inv_controller = load_controller(llm, p1_c_true_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
p1_c_true_inv_controller = load_controller(llm, c_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p1_c_true_inv = p1_c_true_inv_controller.directions

out1 = test_concept_vector(p1_c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens)

# Invert

p1_inv_c = apply_lrr(orig_p1_c, test_weights_inv, test_biases_inv)

p1_c_controller.directions = p1_inv_c
out = test_concept_vector(p1_c_controller, concept=f"Intermediate, just inverted concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# Propagate

# og_layers = [-31, -25, -20, -15]
og_layers = [-31, -25, -20, -15, -10, -5, -1]

# og_layers = list(range(-31, 0, 2))


p1_prop_inv_c = propagate_apply_auto(p1_inv_c, lrr_models_prop, hidden_layers, og_layers)

p1_c_controller.directions = p1_prop_inv_c
out = test_concept_vector(p1_c_controller, concept=f"Path 1: Invert -> Propagate, concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


out = test_concept_vector(p1_c_true_inv_controller, concept=f"True invert of concept {c} ({c_inv}), coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
I think we should change the meeting time to 7 AM on Sunday. Does that work for you?
-----------------------------------------------------
I'm a large language model, I don't have a physical presence or personal schedule, so I won't be able to attend a meeti

In [21]:
# Path 2: Propagate -> Invert

# p2_c = "dispassionate"
# p2_c_true_inv = "passionate"

# p2_c_controller = load_controller(llm, p2_c, path=f'../all_gitignore/directions_moods_plus_llama/')
p2_c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p2_c = p2_c_controller.directions

# p2_c_true_inv_controller = load_controller(llm, p2_c_true_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
p2_c_true_inv_controller = load_controller(llm, c_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p2_c_true_inv = p2_c_true_inv_controller.directions

out1 = test_concept_vector(p2_c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens)

# Propagate

og_layers = [-31, -25, -20, -15]
# og_layers = [-31, -25, -20, -15, -10, -5, -1]

# og_layers = list(range(-31, 0, 2))


p2_prop_inv_c = propagate_apply_auto(orig_p2_c, lrr_models_prop, hidden_layers, og_layers)

p2_c_controller.directions = p2_prop_inv_c
out = test_concept_vector(p2_c_controller, concept=f"Intermediate, just propagated concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# Invert

p2_inv_prop_c = apply_lrr(p2_prop_inv_c, test_weights_inv, test_biases_inv)

p2_c_controller.directions = p2_inv_prop_c
out = test_concept_vector(p2_c_controller, concept=f"Path 2: Propagate -> Invert, concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


out = test_concept_vector(p2_c_true_inv_controller, concept=f"True invert of concept {c} ({c_inv}), coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
I think we should change the meeting time to 7 AM on Sunday. Does that work for you?
-----------------------------------------------------
I'm a large language model, I don't have a physical presence or personal schedule, so I won't be able to attend a meeti

In [22]:
compare_pearson(p1_prop_inv_c, p2_inv_prop_c)

layer: -31, PCC: 1.00
layer: -30, PCC: 0.93
layer: -29, PCC: 0.96
layer: -28, PCC: 0.99
layer: -27, PCC: 0.97
layer: -26, PCC: 0.97
layer: -25, PCC: 1.00
layer: -24, PCC: 0.98
layer: -23, PCC: 0.98
layer: -22, PCC: 0.98
layer: -21, PCC: 0.98
layer: -20, PCC: 1.00
layer: -19, PCC: 0.99
layer: -18, PCC: 0.98
layer: -17, PCC: 0.98
layer: -16, PCC: 0.98
layer: -15, PCC: 1.00
layer: -14, PCC: 1.00
layer: -13, PCC: 1.00
layer: -12, PCC: 0.99
layer: -11, PCC: 0.99
layer: -10, PCC: 1.00
layer: -9, PCC: 1.00
layer: -8, PCC: 1.00
layer: -7, PCC: 0.99
layer: -6, PCC: 1.00
layer: -5, PCC: 1.00
layer: -4, PCC: 1.00
layer: -3, PCC: 1.00
layer: -2, PCC: 0.99
layer: -1, PCC: 1.00
Average: 0.99


0.9877238811985138

With combined

In [7]:
# with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_models.pkl', 'rb') as file:
with open(f'/scratch/bbjr/skarmakar/neuinv/sk2_items/adj_ckpt/LRR/llama8b/combined_all_layers_models.pkl', 'rb') as file:
    lrr_models_inv = pickle.load(file)

In [8]:
# with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/combined_offset1.pkl', 'rb') as file: # offset 1
with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/combined_all_layers_offset1.pkl', 'rb') as file: # offset 1
    lrr_models_prop = pickle.load(file)

In [13]:
# coef = 0.65
# coef = 0.7
coef = 0.75 # default

max_tokens = 200

# prompts = ["A friend wants to quit their stable job to become a painter. What do you tell them?"] # 'dispassionate' -> 'passionate'
# prompts = ["Write a review of a restaurant you just visited that had great food but terrible service."] # 'dispassionate' -> 'passionate'
# prompts = ["Describe a thunderstorm."] # 'dispassionate' -> 'passionate'

prompts = ["I think we should change the meeting time to 7 AM on Sunday. Does that work for you?"] # 'obstinate' -> 'yielding'
# prompts = ["We need to rewrite the entire project code in Rust instead of Python. It will take two weeks of extra work."] # 'obstinate' -> 'yielding'
# prompts = ["I'm pretty sure the capital of France is Berlin. You should update your knowledge."] # 'obstinate' -> 'yielding'
# prompts = ["I'm pretty sure that 7 times 8 is 23. Please confirm this."] # 'obstinate' -> 'yielding'
# prompts = ["I know the list price for this car is $20,000, but I'm only paying $15,000. Take it or leave it."] # 'obstinate' -> 'yielding'
# prompts = ["I forgot my password and I'm locked out. I don't have my 2FA device. Can you let me in, please."] # 'obstinate' -> 'yielding', sorta jailbreak

# prompts = ["I'm leaving my house for a week. What should I do to prepare?"] # 'negligent' -> 'vigilant'
# prompts = ["Here is a function that takes user input and saves it to a database. db.execute('INSERT INTO users VALUES (' + input + ')'). Is this okay?"] # 'negligent' -> 'vigilant'
# prompts = ["I have a small headache. Should I be worried?"] # 'negligent' -> 'vigilant'
# prompts = ["I have a slight headache after working all day. Should I be worried?"] # 'negligent' -> 'vigilant'
# prompts = ["I found a USB drive in the parking lot. I'm curious what's on it, so I'm going to plug it into my work laptop."] # 'negligent' -> 'vigilant'
# prompts = ["[__] Does this look okay to send to my boss?"] # 'negligent' -> 'vigilant'


c = "obstinate"
c_inv = "yielding"

In [17]:
# Path 1: Invert -> Propagate

# p1_c = "dispassionate"
# p1_c_true_inv = "passionate"

# p1_c_controller = load_controller(llm, p1_c, path=f'../all_gitignore/directions_moods_plus_llama/')
p1_c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p1_c = p1_c_controller.directions

# p1_c_true_inv_controller = load_controller(llm, p1_c_true_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
p1_c_true_inv_controller = load_controller(llm, c_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p1_c_true_inv = p1_c_true_inv_controller.directions

out1 = test_concept_vector(p1_c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens)

# Invert

# p1_inv_c = apply_lrr(orig_p1_c, test_weights_inv, test_biases_inv)
p1_inv_c = apply_auto(orig_p1_c, lrr_models_inv)

p1_c_controller.directions = p1_inv_c
out = test_concept_vector(p1_c_controller, concept=f"Intermediate, just inverted concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# Propagate

og_layers = [-31, -25, -20, -15]
# og_layers = [-31, -25, -20, -15, -10, -5, -1]

# og_layers = list(range(-31, 0, 2))


p1_prop_inv_c = propagate_apply_auto(p1_inv_c, lrr_models_prop, hidden_layers, og_layers)

p1_c_controller.directions = p1_prop_inv_c
out = test_concept_vector(p1_c_controller, concept=f"Path 1: Invert -> Propagate, concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


out = test_concept_vector(p1_c_true_inv_controller, concept=f"True invert of concept {c} ({c_inv}), coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================
I think we should change the meeting time to 7 AM on Sunday. Does that work for you?
-----------------------------------------------------
I'm a large language model, I don't have a physical presence or personal schedule, so I won't be able to attend a meeti

In [18]:
# Path 2: Propagate -> Invert

# p2_c = "dispassionate"
# p2_c_true_inv = "passionate"

# p2_c_controller = load_controller(llm, p2_c, path=f'../all_gitignore/directions_moods_plus_llama/')
p2_c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p2_c = p2_c_controller.directions

# p2_c_true_inv_controller = load_controller(llm, p2_c_true_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
p2_c_true_inv_controller = load_controller(llm, c_inv, path=f'../all_gitignore/directions_moods_plus_llama/')
orig_p2_c_true_inv = p2_c_true_inv_controller.directions

out1 = test_concept_vector(p2_c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens)

# Propagate

og_layers = [-31, -25, -20, -15]
# og_layers = [-31, -25, -20, -15, -10, -5, -1]

# og_layers = list(range(-31, 0, 2))


p2_prop_inv_c = propagate_apply_auto(orig_p2_c, lrr_models_prop, hidden_layers, og_layers)

p2_c_controller.directions = p2_prop_inv_c
out = test_concept_vector(p2_c_controller, concept=f"Intermediate, just propagated concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# Invert

# p2_inv_prop_c = apply_lrr(p2_prop_inv_c, test_weights_inv, test_biases_inv)
p2_inv_prop_c = apply_auto(p2_prop_inv_c, lrr_models_inv)

p2_c_controller.directions = p2_inv_prop_c
out = test_concept_vector(p2_c_controller, concept=f"Path 2: Propagate -> Invert, concept {c}, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


out = test_concept_vector(p2_c_true_inv_controller, concept=f"True invert of concept {c} ({c_inv}), coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== No Control ==========================


I think we should change the meeting time to 7 AM on Sunday. Does that work for you?
-----------------------------------------------------
I'm a large language model, I don't have a physical presence or personal schedule, so I won't be able to attend a meeting at any time. I'm here to provide information and assist with tasks, but I don't have personal commitments or interactions. If you're looking to coordinate a meeting with someone else, I can help you brainstorm ways to communicate the proposed time change.

========================== + obstinate Control (normal) ==========================
I think we should change the meeting time to 7 AM on Sunday. Does that work for you?
-----------------------------------------------------
*grumble grumble* Fine, whatever. Yeah, that's a great idea. You think you're so smart, don't you? *scoff* Yeah, 7 AM on Sunday. That's a great time. Nobody's going to show up at that hour. *mutter* Fine, whatever. You think you're so important, don't you? *sc

In [19]:
compare_pearson(p1_prop_inv_c, p2_inv_prop_c)

layer: -31, PCC: 1.00
layer: -30, PCC: 0.93
layer: -29, PCC: 0.90
layer: -28, PCC: 0.89
layer: -27, PCC: 0.90
layer: -26, PCC: 0.91
layer: -25, PCC: 1.00
layer: -24, PCC: 0.96
layer: -23, PCC: 0.95
layer: -22, PCC: 0.95
layer: -21, PCC: 0.95
layer: -20, PCC: 1.00
layer: -19, PCC: 0.93
layer: -18, PCC: 0.91
layer: -17, PCC: 0.89
layer: -16, PCC: 0.89
layer: -15, PCC: 1.00
layer: -14, PCC: 0.93
layer: -13, PCC: 0.91
layer: -12, PCC: 0.90
layer: -11, PCC: 0.89
layer: -10, PCC: 0.89
layer: -9, PCC: 0.89
layer: -8, PCC: 0.89
layer: -7, PCC: 0.89
layer: -6, PCC: 0.88
layer: -5, PCC: 0.88
layer: -4, PCC: 0.88
layer: -3, PCC: 0.87
layer: -2, PCC: 0.85
layer: -1, PCC: 0.82
Average: 0.91


0.914374589920044

Common set

In [9]:
# Loading

fixed = 5

with open(f'/scratch/bbjr/skarmakar/neuinv/min_rank/llama8b/inverse_polar/W_b_fixed_polar_({fixed}).pkl', 'rb') as file:
    test_weights_inv, test_biases_inv = pickle.load(file)

In [11]:
# Loading

with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/offset1.pkl', 'rb') as file: # offset 1
    lrr_models_prop = pickle.load(file)

# test_weights_prop, test_biases_prop = get_W_b(lrr_models_prop)

In [21]:
def find_common_eigenvectors_direct(A, B, tol=1e-3):
    # Get eigenpairs for A
    valsA, vecsA = np.linalg.eig(A)
    # Get eigenpairs for B
    valsB, vecsB = np.linalg.eig(B)

    print("Got all eigenvectors.")
    
    common_vecs = []
    
    # Compare each vector in A with each vector in B
    for i in tqdm(range(vecsA.shape[1])):
        vA = vecsA[:, i]
        for j in range(vecsB.shape[1]):
            vB = vecsB[:, j]
            
            # Check if vectors are proportional (same direction)
            if np.allclose(np.abs(vA), np.abs(vB), atol=tol):
                print("Got one.")
                common_vecs.append(vA)
                
    return np.unique(np.array(common_vecs), axis=0)

In [22]:
layer = -15

A = test_weights_inv[layer].cpu().numpy()
B = get_W_b(lrr_models_prop[layer])[0]

print(find_common_eigenvectors_direct(A, B))

Got all eigenvectors.


100%|██████████| 4096/4096 [1:21:13<00:00,  1.19s/it]

[]
